Before we start, we need to make sure that we have a Kafka cluster running and a topic that produces some streaming data. For simplicity, we will use a single-node Kafka cluster and a topic named `events`. Open the `4.0 events-gen-kafka.ipynb` notebook and execute the cell. This notebook produces an event record every second and put it on a Kafka topic called `events`. 

In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count, to_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [2]:
builder = (SparkSession.builder
           .appName("apply-window-aggregations")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "2g")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder,['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f0249a51-2160-486c-a8de-0af6e30ad48c;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [3]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "events")
      .option("startingOffsets", "earliest")
      .load())

In [4]:
schema = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('event_type', StringType(), True),
    StructField('event_time', StringType(), True),
    StructField('processing_time', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [6]:
df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- user_id: integer (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- event_time: string (nullable = true)
 |    |-- processing_time: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
df = (
     df
    .select(
        col("value.user_id").alias("user_id"),
        col("value.event_type").alias("event_type"),
        col("value.event_time").alias("event_time"),
        col("value.processing_time").alias("processing_time"),
    )
    .withColumn("event_time", to_timestamp(col("event_time"), "MM/dd/yyyy, HH:mm:ss" ))
    .withColumn("processing_time", to_timestamp(col("processing_time"), "MM/dd/yyyy, HH:mm:ss" ))
)

In [10]:
df = (
    df.groupBy(
         window(col("event_time"), "60 minute", "60 minute"), col("event_type"))
        .agg(count(col("user_id")).alias("NumberOfUsers")
    )
)

In [11]:
query = (
     df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |11           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |9            |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |14      

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |11           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |9            |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |14      

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |11           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |9            |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |15      

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |13           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |11           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |9            |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |15      

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |13           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |10           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |12           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |9            |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |15      

...

-------------------------------------------
Batch: 35
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |19           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |13           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |18           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |16           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |24     

-------------------------------------------
Batch: 36
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |19           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |13           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |18           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |17           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |24     

-------------------------------------------
Batch: 37
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |13           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |18           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |17           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |24     

-------------------------------------------
Batch: 38
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |13           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |18           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |18           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |24     

-------------------------------------------
Batch: 39
-------------------------------------------
+------------------------------------------+----------+-------------+
|window                                    |event_type|NumberOfUsers|
+------------------------------------------+----------+-------------+
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|purchase  |20           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|click     |23           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|view      |21           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|view      |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|purchase  |14           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|like      |12           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|like      |18           |
|{2025-02-27 12:00:00, 2025-02-27 13:00:00}|share     |20           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|share     |18           |
|{2025-02-27 13:00:00, 2025-02-27 14:00:00}|click     |24     